#   **LangChain Custom Tool** (Part1)

- 사용자 정의 도구 (Custom Tool) 만들기

---

## 환경 설정 및 준비

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
from glob import glob

from pprint import pprint
import json

---

##  **사용자 정의 도구 (Custom Tool)**


- **사용자 정의 도구**는 개발자가 직접 설계하고 구현하는 **맞춤형 함수나 도구**를 의미

- LLM이 호출할 수 있는 **고유한 기능**을 정의하여 특정 작업에 최적화된 도구 생성 가능

- 개발자는 도구의 **입력값, 출력값, 기능**을 자유롭게 정의하여 유연한 확장성 확보

---

### 1. **`@tool` 데코레이터** 

- **@tool** 데코레이터는 **사용자 정의 도구**를 만드는 가장 기본적인 방법

- 함수의 **이름**과 **독스트링**을 자동으로 도구 정보로 활용

- 간단한 데코레이터 문법으로 **빠른 도구 개발** 가능

`(1) 기본 도구 만들기`

In [ ]:
# 벡터 저장소 로드 
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

chroma_db = Chroma(
    collection_name="db_korean_cosine_metadata",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

In [ ]:
# DB 검색하는 사용자 정의 도구 생성
from langchain_core.tools import tool
from typing import Optional

@tool
def search_database(query: str, k: Optional[int] = 4) -> str:
    """
    데이터베이스에서 주어진 쿼리로 검색을 수행합니다.
    
    Args:
        query: 검색할 텍스트 쿼리
        k: 반환할 결과의 개수 (기본값: 4)
    """
    retriever = chroma_db.as_retriever(search_kwargs={"k": k})
    return retriever.invoke(query)

In [ ]:
# 도구 속성 확인
print(search_database.name)  
print("-" * 100)
print(search_database.description) 
print("-" * 100)
print(search_database.args)  
print("-" * 100)
print(search_database.output_schema.model_json_schema())

In [ ]:
# 도구 실행 
docs = search_database.invoke("리비안은 언제 설립되었나요?")
pprint(docs)

In [ ]:
# LLM 도구 바인딩하여 실행 
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.7)
llm_with_tools = llm.bind_tools([search_database])

# 도구 사용
result = llm_with_tools.invoke("리비안은 언제 설립되었나요?")

# 결과 확인

pprint(result.tool_calls)

In [ ]:
# 도구 사용 (k=2)
result = llm_with_tools.invoke("리비안은 언제 설립되었나요? (2개 문서 검색)")

pprint(result.tool_calls)

`(2) 도구 이름 및 스키마 커스터마이징`

- **@tool 데코레이터**를 사용하여 도구의 속성을 직접 설정 가능
- 도구의 **이름**과 **스키마**를 개발자가 원하는 대로 커스터마이징할 수 있음

In [ ]:
from pydantic import BaseModel, Field

# 도구 입력 스키마 정의 (pydantic 모델 사용)
class ChromaDBInput(BaseModel):
    """ ChromaDB 검색 도구 입력 스키마 """
    query: str = Field(description="검색할 쿼리")
    k: int = Field(4, description="반환할 문서의 개수")

@tool("ChromaDB-Search", args_schema=ChromaDBInput)
def search_database(query: str, k: int = 4) -> str:
    """
    데이터베이스에서 주어진 쿼리로 검색을 수행합니다.
    
    Args:
        query: 검색할 텍스트 쿼리
        k: 반환할 결과의 개수 (기본값: 4)
    """
    retriever = chroma_db.as_retriever(search_kwargs={"k": k})
    return retriever.invoke(query)

In [ ]:
# 도구 속성 확인
print(search_database.name)  
print("-" * 100)
print(search_database.description) 
print("-" * 100)
print(search_database.args)  
print("-" * 100)
print(search_database.output_schema.model_json_schema())

---
### **[실습]**

- 도구 이름과 스키마를 직접 정의하여 도구를 생성합니다. 
- 도구 속성을 확인합니다. 
- 도구를 실행합니다. 
- LLM 모델에 도구를 바인딩하여 도구 호출 결과를 확인합니다. 

In [ ]:
# 여기에 코드를 작성하세요. 


`(3) 비동기 도구 만들기`

- **비동기 도구**는 LangChain에서 `@tool` 데코레이터를 통해 구현 가능
- LangChain은 **동기와 비동기** 두 가지 도구 유형을 모두 지원함
- 비동기 도구는 더 효율적인 **병렬 처리**가 가능한 장점이 있음

In [ ]:
from langchain_core.tools import tool

@tool
async def search_database(query: str, k: int = 4) -> str:
    """
    데이터베이스에서 주어진 쿼리로 검색을 수행합니다.
    
    Args:
        query: 검색할 텍스트 쿼리
        k: 반환할 결과의 개수 (기본값: 4)
    """
    retriever = chroma_db.as_retriever(search_kwargs={"k": k})
    return await retriever.ainvoke(query)

In [ ]:
# 도구 속성 확인
print(search_database.name)
print("-" * 100)
print(search_database.description)
print("-" * 100)
print(search_database.args)
print("-" * 100)
print(search_database.output_schema.model_json_schema())

In [ ]:
# 도구 실행 (비동기)

docs = await search_database.ainvoke("리비안은 언제 설립되었나요?")
pprint(docs)

---
### **[실습]**

- mmr 검색 리트리버를 사용하여 비동기 방식으로 동작하는 도구를 생성합니다. 
- 도구 속성을 확인합니다. 
- 도구를 실행합니다. 

In [ ]:
# 여기에 코드를 작성하세요. 

---

### 2. **StructuredTool** 

- **StructuredTool.from_function**을 통해 도구의 세부 동작을 정의할 수 있음

- 도구의 **실행 방식**과 **응답 처리**를 상세하게 커스터마이징 가능

- 개발자가 원하는 **특정 기능**을 도구에 쉽게 추가할 수 있음

`(1) 입출력 스키마 정의`

In [ ]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field
from typing import Literal
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# 텍스트 분석 입력 스키마 정의
class TextAnalysisInput(BaseModel):
    text: str = Field(description="분석할 텍스트")
    include_sentiment: bool = Field(
        description="감성 분석 포함 여부", 
        default=False
    )

# 감성 분석 출력 스키마 정의
class SentimentOutput(BaseModel):
    sentiment: Literal['positive', 'negative'] = Field(description="감성 분석 결과")

`(2) 도구 작업을 함수로 정의`

In [ ]:
# 텍스트 분석 수행 함수 (동기)
def analyze_text(text: str, include_sentiment: bool = False) -> dict:
    """텍스트를 분석하여 단어 수, 문자 수 등의 정보를 반환합니다."""
    result = {
        "word_count": len(text.split()),
        "char_count": len(text),
        "sentence_count": len(text.split('.')),
    }
    
    if include_sentiment:
        # 감성 분석 수행
        prompt = ChatPromptTemplate.from_messages(
            [
                ("system", "입력된 문장에 대해서 감성 분석을 수행합니다."),
                ("user", "{input}"),
            ]
        )
        llm =  ChatOpenAI(model="gpt-4.1-mini")

        llm_with_structure = llm.with_structured_output(SentimentOutput)

        sentiment_chain = prompt | llm_with_structure

        sentiment = sentiment_chain.invoke({"input": text})

        result["sentiment"] = sentiment.sentiment
        
    return result

# 텍스트 분석 수행 함수 (비동기)
async def analyze_text_async(text: str, include_sentiment: bool = False) -> dict:
    """텍스트 분석의 비동기 버전입니다."""
    return analyze_text(text, include_sentiment)

`(3) 도구 생성단계에서 커스터마이징 가능`

In [ ]:
# 도구 생성
text_analyzer = StructuredTool.from_function(
    func=analyze_text,   # 동기 함수 사용
    name="TextAnalyzer",    # 도구 이름
    description="텍스트의 기본 통계와 선택적으로 감성 분석을 수행합니다.",   # 도구 설명
    args_schema=TextAnalysisInput,   # 입력 스키마
    coroutine=analyze_text_async,    # 비동기 함수 사용
    return_direct=True    # 결과를 직접 반환
)

# 도구 속성 확인
print(text_analyzer.name)
print(text_analyzer.description)
print(text_analyzer.args)
print(text_analyzer.output_schema.model_json_schema())

`(4) 도구 실행`

In [ ]:
# 텍스트 분석 도구 사용
text = "안녕하세요. 오늘은 날씨가 좋네요. 산책하기 좋은 날입니다."

# 동기 호출
result1 = text_analyzer.invoke({
    "text": text,
    "include_sentiment": True
})
print("텍스트 분석 결과:", result1)

# 비동기 호출
result2 = await text_analyzer.ainvoke({
    "text": text,
    "include_sentiment": False
})
print("비동기 텍스트 분석 결과:", result2)

`(5) StructuredTool은 다음과 같은 상황에서 더 적합 (@tool 데코레이터와 차이점)`

- **StructuredTool**은 **기존 함수**를 도구로 쉽게 변환하여 재활용 가능
- 하나의 함수로 **다양한 설정**의 도구를 만들 수 있어 코드 중복을 방지함
- **동기/비동기** 버전을 동시에 지원하여 유연한 실행 환경 제공

In [ ]:
### 1. 기존 함수의 재사용

# 이미 존재하는 함수를 도구로 변환할 때
def existing_function(x: int) -> str:
    return str(x)

# @tool을 사용하려면 함수를 수정해야 함
from langchain_core.tools import tool

@tool 
def modified_function(x: int) -> str:
    """ 숫자 변환 도구 """
    return str(x)

# StructuredTool은 기존 함수를 그대로 사용 가능
tool = StructuredTool.from_function(
    func=existing_function,
    name="convert_number",
    description="숫자를 문자열로 변환합니다.",
)

In [ ]:
### 2. 동일한 함수에 대해 다른 설정의 도구 생성

def multiply(a: int, b: int) -> int:
    return a * b

# 같은 함수로 다른 설정의 도구들을 만들 수 있음
basic_calculator = StructuredTool.from_function(
    func=multiply,
    name="basic_multiply",
    description="기본 곱셈 계산기",
)

advanced_calculator = StructuredTool.from_function(
    func=multiply,
    name="output_multiply", 
    description="결과 출력용",
    return_direct=True   # 결과를 직접 반환
)

# 도구 속성 확인
print(basic_calculator.name)
print(basic_calculator.description)
print(basic_calculator.args)
print(basic_calculator.output_schema.model_json_schema())
print()
print(advanced_calculator.name)
print(advanced_calculator.description)
print(advanced_calculator.args)
print(advanced_calculator.output_schema.model_json_schema())

In [ ]:
from langchain.agents import create_agent

# 도구 실행 에이전트 생성 (return_direct=False)
basic_agent = create_agent(
    model=llm,
    tools=[basic_calculator],
    system_prompt="당신은 수학 계산을 도와주는 AI 어시스턴트입니다."
)

# 도구 실행 에이전트 사용
result = basic_agent.invoke(
    {"messages": [{"role": "user", "content": "2와 3을 곱해줘"}]},
)

pprint(result["messages"])

In [ ]:
# 도구 실행 에이전트 생성 (return_direct=True)
advanced_agent = create_agent(
    model=llm,
    tools=[advanced_calculator],
    system_prompt="당신은 수학 계산을 도와주는 AI 어시스턴트입니다."
)

# 도구 실행 에이전트 사용
result = advanced_agent.invoke(
    {"messages": [{"role": "user", "content": "2와 3을 곱해줘"}]},
)

pprint(result["messages"])

In [ ]:
### 3. 동기/비동기 함수 동시 지원

def sync_func(x: int) -> int:
    return x * 2

async def async_func(x: int) -> int:
    return sync_func(x)

# 동기/비동기 함수를 하나의 도구로 결합
tool = StructuredTool.from_function(
    func=sync_func,  # 동기 함수
    coroutine=async_func,  # 비동기 함수
    name="multiply_by_2",
    description="입력된 숫자에 2를 곱합니다."
)

# 사용
result1 = tool.invoke({"x": 5})  # 동기 호출
result2 = await tool.ainvoke({"x": 5})  # 비동기 호출

print(result1)
print(result2)

---
### **[실습]**

- StructuredTool을 이용하여 mmr 검색 리트리버를 사용하는 도구를 생성합니다. 
- 도구 속성을 확인합니다. 
- 도구를 실행합니다. 

In [ ]:
# 여기에 코드를 작성하세요. 

---

### 3. **Runnable을 도구로 변환** 

- `as_tool` 메소드: **Runnable**을 도구로 변환하여 **복잡한 체인**을 하나의 단위로 관리 가능

- 도구화를 통해 체인에 대한 **명확한 인터페이스**를 제공

- 변환된 도구는 다른 프로젝트나 컴포넌트에서 쉽게 **재사용** 가능

- 체인의 실행 방식을 **표준화**하여 일관된 사용 경험 제공

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 이메일 작성 체인
email_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 전문적인 이메일 작성 도우미입니다."),
    ("human", """
    다음 정보로 이메일을 작성해주세요:
    - 수신자: {recipient}
    - 제목: {subject}
    - 톤: {tone}
    - 추가 요청사항: {requirements}
    """)
])

email_chain = (
    email_prompt 
    | ChatOpenAI(model="gpt-4.1-mini", temperature=0.7) 
    | StrOutputParser()
)

# 이메일 작성 도구로 변환
email_tool = email_chain.as_tool(
    name="email_writer",
    description="전문적인 이메일 작성을 도와주는 도구입니다.",
)

# 도구 속성 변경
email_tool.return_direct = True

# 도구 속성 확인
print(email_tool.name)
print(email_tool.description)
print(email_tool.args)
print(email_tool.return_direct)

In [ ]:
# 도구 실행
email_result = email_tool.invoke({
    "recipient": "team@example.com",
    "subject": "프로젝트 진행 현황 보고",
    "tone": "전문적",
    "requirements": "회의 일정 조율 요청 포함"
})

print(email_result)

In [ ]:
# LLM과 도구 바인딩
llm_with_tools = llm.bind_tools([email_tool])
result = llm_with_tools.invoke("팀에게 프로젝트 진행 현황을 보고하는 이메일을 작성해줘. (전문적 톤, 요구사항: 회의 일정 조율 요청 포함, 수신자: 'team@email.com')")

pprint(result.tool_calls)


In [ ]:
# 도구 호출 결과를 실행
tool_msg = email_tool.invoke(result.tool_calls[0])
print(tool_msg.content)

In [ ]:
from langchain.agents import create_agent

# 도구 실행 에이전트 생성 
email_agent = create_agent(
    model=llm,
    tools=[email_tool],
    system_prompt="당신은 이메일 작성을 도와주는 AI 어시스턴트입니다."
)

# 도구 실행 에이전트 사용
result = email_agent.invoke(
    {"messages": [{"role": "user", "content": "팀에게 프로젝트 진행 현황을 보고하는 이메일을 작성해줘. (전문적 톤, 요구사항: 회의 일정 조율 요청 포함, 수신자: 'team@example.com')"}]},
)

pprint(result["messages"])

---

### **[실습]**

- 검색 결과(문서)와 쿼리 간의 유사도를 CrossEncoderReranker를 사용하여 Re-rank 수행 (k:10 -> top_n:3)
- 검색 결과를 포맷팅하여 출력하는 Runnable 체인을 구성
- Runnable 체인을 도구로 변환 

In [ ]:
from langchain_community.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_core.runnables import RunnableLambda

# 1. CrossEncoderReranker 설정
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
compressor = CrossEncoderReranker(model=model, top_n=3)

# 2. 기본 retriever 생성 (k=10)
base_retriever = chroma_db.as_retriever(search_kwargs={"k": 10})

# 3. Reranker가 적용된 retriever 생성
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

# 4. 검색 결과를 포맷팅하는 함수
def format_docs(docs):
    """검색된 문서를 포맷팅하여 반환"""
    formatted = []
    for i, doc in enumerate(docs, 1):
        formatted.append(f"[문서 {i}]")
        formatted.append(f"내용: {doc.page_content}")
        formatted.append(f"메타데이터: {doc.metadata}")
        formatted.append("-" * 80)
    return "\n".join(formatted)

# 5. Runnable 체인 구성
search_and_format_chain = compression_retriever | RunnableLambda(format_docs)

# 6. Runnable 체인을 도구로 변환
reranked_search_tool = search_and_format_chain.as_tool(
    name="reranked_database_search",
    description="데이터베이스에서 검색을 수행하고 CrossEncoder를 사용하여 상위 3개 결과를 재순위화합니다. 포맷팅된 결과를 반환합니다.",
)

# 도구 속성 설정
reranked_search_tool.return_direct = True

# 도구 속성 확인
print("도구 이름:", reranked_search_tool.name)
print("-" * 100)
print("도구 설명:", reranked_search_tool.description)
print("-" * 100)
print("도구 인자:", reranked_search_tool.args)
print("-" * 100)
print("직접 반환:", reranked_search_tool.return_direct)

In [ ]:
# 도구 실행 테스트
result = reranked_search_tool.invoke({"input": "리비안은 언제 설립되었나요?"})
print(result)